# TAMPO Algorithms — Colab Test Run Notebook

This notebook tests the currently implemented TAMPO meta-RL algorithms (TAMPO-GCN, TAMPO-GAT, and TAMPO-LSTM).
Run cells **top to bottom** in order.

## 0. Setup — Clone repo & install dependencies

In [ ]:
# ── 0a. Clone the repo ────────────────────────────────────────────
!git clone -b Vikkesh https://github.com/vikkesh/tampo.git tampo
%cd tampo

In [ ]:
# If you want to pull latest commits from the Vikkesh branch

# 1. Fetch the latest changes from GitHub
!git fetch origin

# 2. Checkout gat-pyg branch
!git checkout Vikkesh

# 3. Hard reset to exactly match the latest code on that branch
!git reset --hard origin/Vikkesh

In [ ]:
!nvidia-smi

In [2]:
# ── 0b. Install all dependencies ─────────────────────────────────
!pip install -r requirements.txt

In [3]:
# ── 0c. CUDA config + Verify imports ────────────────────────────────────────
# PYTORCH_CUDA_ALLOC_CONF must be set BEFORE CUDA operations start.
# Setting it here (same cell as first torch use) guarantees it survives
# any kernel restart triggered by pip install above.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch, gymnasium as gym, yaml, json, numpy as np
from torch_geometric.data import Batch, Data

print('torch           :', torch.__version__)
print('gymnasium       :', gym.__version__)
print('CUDA available  :', torch.cuda.is_available())
print('ALLOC_CONF      :', os.environ['PYTORCH_CUDA_ALLOC_CONF'])
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    free_gb  = torch.cuda.mem_get_info()[0] / 1024**3
    total_gb = props.total_memory / 1024**3
    print(f'GPU             : {props.name}')
    print(f'VRAM            : {total_gb:.1f} GB total | {free_gb:.1f} GB free')


## 1. Unit Test — DAGEncoder with Bi-GCN and Bi-GATv2 paths

Verifies that both encoders:
- Accept a PyG Batch of **variable-size** graphs
- Produce a context vector of the correct shape
- Produce **identical output shapes** (the only variable is the conv operator)

In [ ]:
import sys, os
sys.path.insert(0, '.')

import torch
from torch_geometric.data import Batch, Data
from algorithms.rl.tampo import DAGEncoder

TASK_FEATURE_DIM   = 6
HIDDEN_DIM         = 128
SERVER_FEATURE_DIM = 20

# Three graphs with DIFFERENT node counts (8, 12, 20)
graphs = []
for n in [8, 12, 20]:
    x = torch.rand(n, TASK_FEATURE_DIM)
    src = torch.arange(0, n - 1)
    dst = torch.arange(1, n)
    edge_index = torch.stack([src, dst], dim=0)
    graphs.append(Data(x=x, edge_index=edge_index, num_nodes=n))

batch = Batch.from_data_list(graphs)
task_features_dummy = torch.zeros(3, 20, TASK_FEATURE_DIM)
server_features     = torch.rand(3, SERVER_FEATURE_DIM)

results = {}
for enc_type in ['gcn', 'gat']:
    encoder = DAGEncoder(
        task_feature_dim=TASK_FEATURE_DIM,
        hidden_dim=HIDDEN_DIM,
        num_layers=2,
        encoder_type=enc_type,
        server_feature_dim=SERVER_FEATURE_DIM
    )
    encoder.eval()
    with torch.no_grad():
        encoded_tasks, context = encoder(
            task_features=task_features_dummy,
            graph_batch=batch,
            server_features=server_features
        )
    assert context.shape       == (3, HIDDEN_DIM * 2), f"[{enc_type}] Bad context shape: {context.shape}"
    assert encoded_tasks.shape == (3, 20, HIDDEN_DIM * 2), f"[{enc_type}] Bad encoded_tasks shape: {encoded_tasks.shape}"
    # RC#2 fix: encoded_tasks must be NON-ZERO (was all-zeros before fix)
    assert encoded_tasks.abs().max().item() > 1e-9, f"[{enc_type}] encoded_tasks is all-zeros — RC#2 fix not applied!"
    assert not torch.isnan(context).any(),       f"[{enc_type}] NaN in context"
    assert not torch.isnan(encoded_tasks).any(), f"[{enc_type}] NaN in encoded_tasks"
    results[enc_type] = {'context': context.shape, 'encoded_tasks': encoded_tasks.shape}
    print(f"PASS [{enc_type.upper()}] context={context.shape}  encoded_tasks={encoded_tasks.shape}  max_abs={encoded_tasks.abs().max():.4f}")

# Shape equality check — critical for fair comparison
assert results['gcn']['context']       == results['gat']['context'],       "MISMATCH: context shapes differ"
assert results['gcn']['encoded_tasks'] == results['gat']['encoded_tasks'], "MISMATCH: encoded_tasks shapes differ"
print("\nPASS — GCN and GAT produce identical output shapes, and encoded_tasks is non-zero ✓")


## 2. Generate the Golden Test Dataset

> ⚠️ Run **once only**.  Never re-run between algorithm comparisons.

In [5]:
!python utils/generate_test_dataset.py --num_dags 20 --output data/test_dags.json

import json
with open("data/test_dags.json") as f:
    ds = json.load(f)
print(f"Dataset contains {len(ds)} entries")
print("First entry keys:", list(ds[0].keys()))

## 2.5 Verification — Physics and Reward Overhaul

Confirms that the environment correctly implements dynamic server loads (Makespan) and diverse, context-sensitive rewards.

In [6]:
# ── 2.5 Verification — Physics and Reward Overhaul ──────────────────────────
# Verifies the convergence fixes applied on 2026-07-06:
#   RC#3  — reward scale ±1.0 (was ±5.0)
#   RC#15 — kappa 1e-23 (was 1e-28, making energy signal learnable)
#   RC#1  — reward is positive when offloading is better than local

import sys, yaml, json, numpy as np
sys.path.insert(0, '.')
from env.base_offloading_env import TaskOffloadingEnv

with open('configs/default_config.yaml') as f:
    fc = yaml.safe_load(f)
cfg = {}
for sec in ('system','computing','energy','network','tasks'):
    cfg.update(fc.get('environment', fc).get(sec, {}))
cfg['reward'] = fc.get('environment', fc).get('reward', {})

env = TaskOffloadingEnv(cfg)

with open('data/test_dags.json') as f:
    dags = json.load(f)

dag = dags[0]
state = env.reset(task_graph=dag, preference_vector=np.array([0.5, 0.5]))

print('--- A: Reward scale check (RC#3) ---')
print(f'kappa in config: {env.kappa:.2e}  (should be ~1e-23)')

rewards = []
done = False
step = 0
while not done and step < dag['num_tasks']:
    action = step % env.action_space.n
    obs, reward, done, info = env.step(action)
    rewards.append(reward)
    step += 1

rewards_arr = np.array(rewards)
print(f'Reward range: [{rewards_arr.min():.4f}, {rewards_arr.max():.4f}]')
assert rewards_arr.max() <= 1.01 and rewards_arr.min() >= -1.01, (
    f'FAIL: Rewards outside ±1.0 range! Got [{rewards_arr.min():.4f}, {rewards_arr.max():.4f}]'
)
print('PASS — Reward clipped to ±1.0  ✓')
print(f'Makespan: {env.total_delay:.4f}s  Energy: {env.total_energy:.8f}J')

print()
print('--- B: Energy magnitude check (RC#15) ---')
local_energy_per_node = env.kappa * 1e9 * (env.local_freq ** 2)  # typical 1GHz node
cloud_energy_per_node = (1e6 / env.bandwidth_up) * env.cloud_power_tx
ratio = cloud_energy_per_node / max(local_energy_per_node, 1e-30)
print(f'Local energy (1GHz node): {local_energy_per_node:.3e} J')
print(f'Cloud tx energy (1MB):    {cloud_energy_per_node:.3e} J')
print(f'Ratio cloud/local:        {ratio:.1f}x')
assert ratio < 1e6, (
    f'FAIL: Energy ratio is {ratio:.1e}x — kappa fix (RC#15) not applied correctly'
)
print('PASS — Energy magnitudes within learnable range  ✓')

print()
print('--- C: Server state dynamics check ---')
state = env.reset(task_graph=dag, preference_vector=np.array([0.5, 0.5]))
snapshots = [env.server_available.copy()]
done = False; step = 0
while not done and step < dag['num_tasks']:
    obs, reward, done, info = env.step(step % env.action_space.n)
    snapshots.append(env.server_available.copy())
    step += 1
all_same = all(np.allclose(snapshots[0], s) for s in snapshots[1:])
print('FAIL — server state is static.' if all_same else 'PASS — server state changes dynamically  ✓')


## 3. Quick Training Smoke Test — TAMPO-GCN and TAMPO-LSTM (1 iteration)

Checks that the full training loop executes without shape errors for both algorithms.

In [ ]:
import yaml, sys, os, random
sys.path.insert(0, '.')

with open('configs/default_config.yaml') as f:
    full_config = yaml.safe_load(f)

from env.base_offloading_env import TaskOffloadingEnv
from algorithms.rl.tampo import TAMPOFramework
from utils.training_setup import load_training_graphs, build_env_task_list

cfg = {}
env_cfg = full_config.get('environment', full_config)
for sec in ('system','computing','energy','network','tasks'):
    cfg.update(env_cfg.get(sec, {}))
cfg['reward'] = env_cfg.get('reward', {})

env = TaskOffloadingEnv(cfg)

# ── Training Graph Pool ────────────────────────────────────────────────────────
# ALL 9 available node sizes are used for training (10–50 in steps of 5).
# Zero-shot generalisation is tested via UNSEEN GRAPH TOPOLOGIES in test_dags.json,
# not unseen sizes — every graph in test_dags.json is a brand-new random DAG
# that was never loaded during this training loop.
#
# Pool: 20 graphs × 9 sizes = 180 total training graphs
task_graphs   = load_training_graphs()           # configured in utils/training_setup.py
tasks_for_env = build_env_task_list(task_graphs)
env.load_task_dataset(tasks_for_env)

os.makedirs('models', exist_ok=True)

for enc_type in ['gcn', 'gat', 'lstm']:
    print(f'\n{"="*42}')
    print(f'  Smoke Test — TAMPO-{enc_type.upper()}')
    print(f'{"="*42}')
    tampo_cfg = {**full_config['training'], **full_config['algorithms']['tampo']}
    tampo_cfg['encoder_type'] = enc_type

    framework = TAMPOFramework(env, tampo_cfg)
    framework.train(num_iterations=1, meta_batch_size=2,
                    checkpoint_path=f'models/tampo_{enc_type}_checkpoint.pth')
    framework.save(f'models/tampo_{enc_type}_checkpoint.pth')
    print(f'SMOKE TEST PASSED — TAMPO-{enc_type.upper()} checkpoint saved.')

## 3.5 Complete Training — TAMPO-GCN and TAMPO-LSTM

In [ ]:
# ── Full Training Run ────────────────────────────────────────────────────────
#
# CHECKPOINT RESUME: This cell automatically resumes from the last saved
# checkpoint. If models/tampo_gcn_checkpoint.pth already exists (e.g. from a
# previous Colab session), training continues from where it left off.
# To start fresh: delete the .pth files in the models/ folder first.
#
# FAIRNESS NOTE: All three encoders (GCN, GAT, LSTM) use identical
# hyperparameters — same inner_steps, same meta_batch_size, same episodes.
# This ensures the benchmark comparison is valid.
#
# IF LSTM OOMS: lower LSTM_META_BATCH below (e.g. 8). This reduces the number
# of tasks sampled per outer update (gradient variance), NOT the number of
# inner adaptation steps. Inner_steps stays equal across all three models,
# so the comparison fairness is preserved.
#
# num_iterations=75   → solid first run; re-run this cell to add more on top
# meta_batch_size=15  → 15 task DAGs sampled per outer-loop update

import os

NUM_ITERATIONS       = 75   # iterations THIS session (stacks on any checkpoint)
META_BATCH_SIZE      = 15   # same for GCN and GAT
LSTM_META_BATCH_SIZE = 15   # reduce to 8 if LSTM OOMs (inner_steps stays equal)

encoder_batch = {
    'gcn':  META_BATCH_SIZE,
    'gat':  META_BATCH_SIZE,
    'lstm': LSTM_META_BATCH_SIZE,
}

for enc_type in ['gcn', 'gat', 'lstm']:
    ckpt_path   = f'models/tampo_{enc_type}_checkpoint.pth'
    batch_size  = encoder_batch[enc_type]

    print(f'\n{"="*55}')
    print(f'  Full Training — TAMPO-{enc_type.upper()}')
    print(f'  meta_batch_size = {batch_size}')
    if os.path.exists(ckpt_path):
        print(f'  Resuming from: {ckpt_path}')
    else:
        print(f'  Starting from scratch')
    print(f'{"="*55}')

    tampo_cfg = {**full_config['training'], **full_config['algorithms']['tampo']}
    tampo_cfg['encoder_type'] = enc_type

    # model_path → auto-loads checkpoint + restores iteration count
    framework = TAMPOFramework(env, tampo_cfg, model_path=ckpt_path)

    framework.train(
        num_iterations=NUM_ITERATIONS,
        meta_batch_size=batch_size,
        checkpoint_path=ckpt_path
    )
    framework.save(ckpt_path)

    total_done = framework.training_history['iterations']
    print(f'\n✓  TAMPO-{enc_type.upper()} — {total_done} total iterations done. Checkpoint saved.')


## 4. Run Benchmark on Trained Models

Compares both trained algorithms against the same test dataset.

In [ ]:
import os, sys
sys.path.insert(0, '.')

# ── Which checkpoint file does what? ─────────────────────────────────────────
# Training produces TWO files per encoder:
#
#   tampo_gcn_checkpoint.pth        ← RESUME checkpoint
#       Saved every 10 iterations + at session end.
#       Use as model_path= to continue training from where you left off.
#       This file is NEVER touched by the benchmark step.
#
#   tampo_gcn_checkpoint_best.pth   ← EVALUATION checkpoint
#       Saved whenever meta-loss hits a new all-time minimum.
#       Best-generalising weights — used for benchmarking via --use_best.

# Confirm which files exist before running
for enc_type in ['gcn', 'gat', 'lstm']:
    best = f'models/tampo_{enc_type}_checkpoint_best.pth'
    reg  = f'models/tampo_{enc_type}_checkpoint.pth'
    if os.path.exists(best):
        print(f'[{enc_type.upper()}] Will evaluate: {best}')
    elif os.path.exists(reg):
        print(f'[{enc_type.upper()}] No _best.pth — will fall back to: {reg}')
    else:
        print(f'[{enc_type.upper()}] WARNING: no checkpoint found — will use untrained weights.')

print()

# --use_best loads _best.pth for each encoder.
# Falls back to the regular checkpoint if _best.pth doesn't exist.
# Does NOT overwrite or modify any checkpoint files.
!python benchmark.py \
    --algorithms TAMPO_GCN TAMPO_GAT TAMPO_LSTM \
    --checkpoint_dir models/ \
    --dataset_path data/test_dags.json \
    --output_dir results/ \
    --use_best


## 5. Download Results

In [ ]:
import os
import glob
from google.colab import files

# Find the most recently created run directory
run_dirs = sorted(glob.glob('results/run_*'), reverse=True)
if run_dirs:
    latest_run = run_dirs[0]
    for fname in ['comparison_bar.png', 'pareto_front.png', 'benchmark_results.csv']:
        path = os.path.join(latest_run, fname)
        if os.path.exists(path):
            files.download(path)
            print(f'Downloaded {path}')
        else:
            print(f'Not found: {path}')
else:
    print('No results found. Run benchmark first.')


## 6. Download Models
Zip and download the trained model checkpoints.

In [ ]:
import shutil
from google.colab import files
import os

# Define the folder to zip and the output name
folder_to_zip = 'models'
output_filename = 'models.zip'

if os.path.exists(folder_to_zip):
    # Create zip archive
    shutil.make_archive('models', 'zip', folder_to_zip)
    
    # Trigger download
    files.download(output_filename)
    print(f'Successfully zipped and started download for {output_filename}')
else:
    print(f'Error: Folder "{folder_to_zip}" not found.')

## Backing up results andmodels folders to Gdrive

In [ ]:
# ── 8. Backup Results to Google Drive (WSL/IDE Fallback) ───────
from google.colab import drive
import shutil
import os
from datetime import datetime

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define destination with timestamp parent folder
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
drive_destination = f'/content/drive/MyDrive/Task_Offloading_Results/backup_{timestamp}'
os.makedirs(drive_destination, exist_ok=True)

# 3. Copy results and models folders
shutil.copytree('results', f'{drive_destination}/results', dirs_exist_ok=True)
shutil.copytree('models', f'{drive_destination}/models', dirs_exist_ok=True)
print(f'✅ Successfully backed up results and models to:{drive_destination}')